# CONFIG

In [1]:
import os
import pandas as pd
import numpy as np
import cv2

from maikol_utils.print_utils import print_separator
import matplotlib.pyplot as plt


pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)

%load_ext autoreload
%autoreload 2
%matplotlib inline

In [3]:
from src.config import Configuration

CONFIG = Configuration(
    crop_size=24,
    stride=4 # We aer not going to use them all, we compute a small part so no need to pixel wise
)

# Generate all no faces

## Load images
The labels to keep are generated from [this scrip](generate_labels.py)

In [4]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F

from maikol_utils.file_utils import load_json

to_keep_labels = load_json(CONFIG.dataset_classes_path)

# Download the validation set
dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=to_keep_labels,
    max_samples=5000,
)
dataset = dataset.filter_labels("ground_truth", F("label").is_in(to_keep_labels))

/home/turbotowerlnx/Documents/Master/BIOM/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading output from data/dataset_classes.json...
Necessary images already downloaded
Existing download of split 'train' is sufficient
Loading existing dataset 'open-images-v7-train-5000'. To reload from disk, either delete the existing dataset or provide a custom `dataset_name` to use


## Generate all crops

We have 136_965 face images, since we are loading 5_000, take  $136.965/5.000 = 27.2 \to 28 \times 2 = 56$ crops from each image to genesrge a non-faces dataset.

In [5]:
from tqdm import tqdm
from src.data import get_all_image_crops
rng = np.random.default_rng(CONFIG.seed)

CROPS_PER_IMAGE = 28*2
filepaths = list(dataset.values("filepath"))

all_crops = []
for img_path in tqdm(filepaths):
    crops = get_all_image_crops(CONFIG, img_path=img_path)
    all_crops.extend(rng.choice(crops, size=CROPS_PER_IMAGE, replace=False))

all_crops = [i['img'] for i in all_crops]


100%|██████████| 5000/5000 [11:07<00:00,  7.49it/s]


In [6]:
for img, idx in tqdm(zip(all_crops, range(len(all_crops)))):
    path = os.path.join(CONFIG.crops_path, f"{idx}.jpg")
    cv2.imwrite(path, img)

0it [00:00, ?it/s]

280000it [00:03, 85284.38it/s]


# Create partitions